# TraderOptimizer Batch Results

This notebook summarizes the latest `TraderOptimizer` batch run and shows the generated best configuration. It reads the batch artifacts from `runs/batch_existing`, ranks every optimized strategy by objective value and realized return, and displays the best config JSON for inspection.

The batch run used the PostgreSQL `historical_bars` table with 8 Optuna trials per discovered strategy config.

In [1]:
from __future__ import annotations

from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'trader_optimizer').is_dir():
            return candidate
    raise RuntimeError('Could not locate TraderOptimizer repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RUN_DIR = REPO_ROOT / 'runs' / 'batch_existing'
SUMMARY_PATH = RUN_DIR / 'batch_summary.json'

batch = json.loads(SUMMARY_PATH.read_text())
items = []
skipped = []

for result in batch['results']:
    if result['status'] != 'ok':
        skipped.append(result)
        continue
    summary_path = Path(result['summary'])
    summary = json.loads(summary_path.read_text())
    metrics = summary['metrics']['all']
    best_config = json.loads(Path(result['best_config']).read_text())
    items.append({
        'name': result['name'],
        'strategy_type': result['strategy_type'],
        'variant': result['variant'],
        'symbols': ','.join(result['symbols']),
        'best_value': result['best_value'],
        'return_pct': metrics['return_pct'],
        'net_pnl': metrics['net_pnl'],
        'fills': metrics['fills'],
        'final_position_value': metrics['final_position_value'],
        'best_config_path': result['best_config'],
        'summary_path': result['summary'],
        'source_config': result['source_config'],
        'best_config': best_config,
        'summary': summary,
    })

items_by_value = sorted(items, key=lambda row: row['best_value'], reverse=True)
items_by_return = sorted(items, key=lambda row: row['return_pct'], reverse=True)
best = items_by_value[0]

display(Markdown(f"Loaded **{len(items)} optimized configs** from `{RUN_DIR}`. Skipped: **{len(skipped)}**."))
display(Markdown(f"Best objective: **{best['name']}** (`{best['strategy_type']}` / `{best['variant']}`), value **{best['best_value']:.6f}**."))

Loaded **32 optimized configs** from `/Users/vrajpandya/.openclaw/workspace/Trader/TraderOptimizer/runs/batch_existing`. Skipped: **0**.

Best objective: **constant_step_offset_mu_tws** (`ConstantStepOffset` / `CSO`), value **0.071808**.

In [2]:
def render_table(rows, columns):
    header = ''.join(f'<th>{html.escape(label)}</th>' for _, label in columns)
    body_rows = []
    for row in rows:
        cells = []
        for key, _ in columns:
            value = row[key]
            if isinstance(value, float):
                if key.endswith('pct'):
                    value = f'{value:.2%}'
                else:
                    value = f'{value:,.4f}'
            cells.append(f'<td>{html.escape(str(value))}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    return HTML('''
    <style>
      table.trader-opt { border-collapse: collapse; width: 100%; font-size: 13px; }
      table.trader-opt th { text-align: left; border-bottom: 2px solid #333; padding: 6px; }
      table.trader-opt td { border-bottom: 1px solid #ddd; padding: 6px; vertical-align: top; }
      table.trader-opt tr:nth-child(even) { background: #fafafa; }
    </style>
    <table class="trader-opt"><thead><tr>''' + header + '''</tr></thead><tbody>''' + ''.join(body_rows) + '''</tbody></table>
    ''')


columns = [
    ('name', 'strategy'),
    ('strategy_type', 'type'),
    ('variant', 'variant'),
    ('symbols', 'symbols'),
    ('best_value', 'objective'),
    ('return_pct', 'return'),
    ('net_pnl', 'net pnl'),
    ('fills', 'fills'),
]

display(Markdown('## Top 12 by Optuna objective'))
display(render_table(items_by_value[:12], columns))

display(Markdown('## Top 12 by simulated return'))
display(render_table(items_by_return[:12], columns))

## Top 12 by Optuna objective

strategy,type,variant,symbols,objective,return,net pnl,fills
constant_step_offset_mu_tws,ConstantStepOffset,CSO,MU,0.0718,16.39%,"4,938.7700",44
ts002_ema_cross_tsla_postgres,TechnicalSignal,TS-002,TSLA,0.0718,7.51%,475.6800,40
ts003_bollinger_breakout_amd_postgres,TechnicalSignal,TS-003,AMD,0.0535,5.59%,39.1800,4
constant_step_offset_amd_tws_postgres,ConstantStepOffset,CSO,AMD,0.0482,7.94%,731.1600,10
constant_step_offset_amd_tws,ConstantStepOffset,CSO,AMD,0.0411,8.20%,454.3200,14
constant_step_offset_tsla_tws,ConstantStepOffset,CSO,TSLA,0.0284,5.61%,"2,164.3875",35
constant_step_offset_nvda_tws,ConstantStepOffset,CSO,NVDA,0.0278,4.71%,49.9300,10
constant_step_offset_aapl_tws,ConstantStepOffset,CSO,AAPL,0.0222,3.31%,"2,729.6750",31
constant_step_offset_aapl_tws_postgres,ConstantStepOffset,CSO,AAPL,0.0212,3.06%,61.4275,13
constant_step_offset_avgo_tws,ConstantStepOffset,CSO,AVGO,0.0199,2.83%,788.7900,31


## Top 12 by simulated return

strategy,type,variant,symbols,objective,return,net pnl,fills
constant_step_offset_mu_tws,ConstantStepOffset,CSO,MU,0.0718,16.39%,"4,938.7700",44
constant_step_offset_amd_tws,ConstantStepOffset,CSO,AMD,0.0411,8.20%,454.3200,14
constant_step_offset_amd_tws_postgres,ConstantStepOffset,CSO,AMD,0.0482,7.94%,731.1600,10
ts002_ema_cross_tsla_postgres,TechnicalSignal,TS-002,TSLA,0.0718,7.51%,475.6800,40
constant_step_offset_tsla_tws,ConstantStepOffset,CSO,TSLA,0.0284,5.61%,"2,164.3875",35
ts003_bollinger_breakout_amd_postgres,TechnicalSignal,TS-003,AMD,0.0535,5.59%,39.1800,4
constant_step_offset_nvda_tws,ConstantStepOffset,CSO,NVDA,0.0278,4.71%,49.9300,10
constant_step_offset_aapl_tws,ConstantStepOffset,CSO,AAPL,0.0222,3.31%,"2,729.6750",31
constant_step_offset_aapl_tws_postgres,ConstantStepOffset,CSO,AAPL,0.0212,3.06%,61.4275,13
constant_step_offset_intc_tws,ConstantStepOffset,CSO,INTC,0.0141,2.95%,314.4600,14


In [3]:
family_rows = []
for strategy_type in sorted({item['strategy_type'] for item in items}):
    subset = [item for item in items if item['strategy_type'] == strategy_type]
    family_rows.append({
        'strategy_type': strategy_type,
        'count': len(subset),
        'best_strategy': max(subset, key=lambda row: row['best_value'])['name'],
        'best_value': max(row['best_value'] for row in subset),
        'avg_return_pct': sum(row['return_pct'] for row in subset) / len(subset),
    })

display(Markdown('## Strategy family coverage'))
display(render_table(family_rows, [
    ('strategy_type', 'type'),
    ('count', 'configs'),
    ('best_strategy', 'best strategy'),
    ('best_value', 'best objective'),
    ('avg_return_pct', 'avg return'),
]))

## Strategy family coverage

type,configs,best strategy,best objective,avg return
ConstantStepOffset,23,constant_step_offset_mu_tws,0.0718,2.60%
MovingAverageCross,2,moving_average_cross_aapl_tws_postgres,-0.2446,1.81%
PortfolioAllocation,3,qs001_volatility_targeting_top_stocks_postgres,-0.0043,-4.34%
TechnicalSignal,4,ts002_ema_cross_tsla_postgres,0.0718,3.21%


In [4]:
chart_rows = items_by_return[:18]
width = 900
row_height = 26
left_pad = 260
right_pad = 90
height = 40 + row_height * len(chart_rows)
min_ret = min(row['return_pct'] for row in chart_rows)
max_ret = max(row['return_pct'] for row in chart_rows)
span = max(max_ret - min_ret, 1e-9)

bars = []
for idx, row in enumerate(chart_rows):
    y = 30 + idx * row_height
    normalized = (row['return_pct'] - min_ret) / span
    bar_w = max(2, int(normalized * (width - left_pad - right_pad)))
    color = '#0f766e' if row['return_pct'] >= 0 else '#b91c1c'
    label = html.escape(row['name'])
    value = f"{row['return_pct']:.2%}"
    bars.append(f'<text x="8" y="{y + 16}" font-size="12">{label}</text>')
    bars.append(f'<rect x="{left_pad}" y="{y}" width="{bar_w}" height="18" fill="{color}" opacity="0.85" />')
    bars.append(f'<text x="{left_pad + bar_w + 8}" y="{y + 14}" font-size="12">{value}</text>')

svg = f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">' + ''.join(bars) + '</svg>'
display(Markdown('## Simulated return chart'))
display(HTML(svg))

## Simulated return chart

In [5]:
display(Markdown('## Best generated configuration'))
display(Markdown(f"Best config path: `{best['best_config_path']}`"))
display(Markdown(f"Source config: `{best['source_config']}`"))
display(HTML('<pre style="white-space: pre-wrap; font-size: 12px; border: 1px solid #ddd; padding: 12px; background: #fafafa;">' + html.escape(json.dumps(best['best_config'], indent=2)) + '</pre>'))

## Best generated configuration

Best config path: `/Users/vrajpandya/.openclaw/workspace/Trader/TraderOptimizer/runs/batch_existing/constant_step_offset_mu_tws/best_config.json`

Source config: `/Users/vrajpandya/.openclaw/workspace/Trader/TraderCore/TraderLogicConfigs/TraderLab/configs/backtests/ibkr_stock_stress/constant_step_offset_mu_tws.json`